In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')  # NOTA: due punti, non 'memory:'
cursor = conn.cursor()

# DATI DI PRODUZIONE MINERARIA (dati sporchi)
data_produzione = {
    'ID_Mese': ['GEN23', 'feb23', 'MAR23', 'apr23', 'MAG23', 'giu23', 'LUG23', 'ago23', 'SET23', 'ott23', 'NOV23', 'dic23'],
    'Data': ['2023-01-01', '01/02/2023', '2023.03.01', '01-04-2023', '2023/05/01', '2023-06-01', '01/07/2023', '2023.08.01', '01-09-2023', '2023/10/01', '2023-11-01', '31-12-2023'],
    'Sito_estrazione': ['Nord', 'nord', 'SUD', 'Sud', 'Est', 'EST', 'ovest', 'Ovest', 'Nord', 'SUD', 'est', 'Nord'],
    'Minerali_estratti': ['Oro,Argento', 'oro,argento', 'Rame,Oro', 'Oro', 'Argento,Rame', 'Oro,Argento,Rame', 'Rame', 'Oro', 'Argento', 'Oro,Rame', 'Argento', 'Oro,Argento'],
    'Tonni_estratte': ['1250', '1350', '1100', '950', '880', '1020', '980', '1150', '1250', '1080', '1120', '1300'],
    'Oro_grammi': ['2.5', '2.8', '1.8', '2.1', '0', '2.2', '0', '2.4', '0', '2.0', '0', '2.3'],
    'Argento_grammi': ['15', '18', '0', '12', '8.5', '20', '0', '0', '25', '0', '15', '22'],
    'Rame_kg': ['120', '130', '150', '0', '90', '180', '200', '0', '0', '160', '0', '140'],
    'Ore_lavorate': ['720', '700', '680', '650', '620', '690', '640', '710', '680', '660', '670', '700'],
    'Note': ['Produzione regolare', 'Ritardo consegne', 'Manutenzione impianti', 'Nuovo fronte', 'Messa in sicurezza', 'Picco produttivo', 'Guasto macchinari', 'Produzione regolare', 'Sciopero', 'Record estrazione', 'Ferma per inventario', 'Fine anno']
}

# Carica in SQLite
df_produzione = pd.DataFrame(data_produzione)
df_produzione.to_sql('Produzione_Grezza', conn, index=False, if_exists='replace')

print("=== DATI GREZZI PRODUZIONE MINERARIA ===")
print(df_produzione)

=== DATI GREZZI PRODUZIONE MINERARIA ===
   ID_Mese        Data Sito_estrazione Minerali_estratti Tonni_estratte  \
0    GEN23  2023-01-01            Nord       Oro,Argento           1250   
1    feb23  01/02/2023            nord       oro,argento           1350   
2    MAR23  2023.03.01             SUD          Rame,Oro           1100   
3    apr23  01-04-2023             Sud               Oro            950   
4    MAG23  2023/05/01             Est      Argento,Rame            880   
5    giu23  2023-06-01             EST  Oro,Argento,Rame           1020   
6    LUG23  01/07/2023           ovest              Rame            980   
7    ago23  2023.08.01           Ovest               Oro           1150   
8    SET23  01-09-2023            Nord           Argento           1250   
9    ott23  2023/10/01             SUD          Oro,Rame           1080   
10   NOV23  2023-11-01             est           Argento           1120   
11   dic23  31-12-2023            Nord       Oro,Argento   

In [8]:
# Query di pulizia
q = """
SELECT 
 -- Pulizia ID_Mese (-> gen23')
UPPER(TRIM(ID_mese)) AS id_mese,

 -- Pulizia Data (-> formato unico YYYY-MM-DD)
CASE
WHEN Data LIKE '__/__/____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '__-__-____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '%.%' THEN 
    REPLACE(Data, '.', '-')
WHEN Data LIKE '%/%' THEN 
    REPLACE(Data, '/', '-')
ELSE Data
END AS data,

 -- Sito_estrazione (-> NORD)
UPPER(TRIM(Sito_estrazione)) AS sito_estrazione,

 -- Tonni_estratte (-> 1250.0)
CAST(TRIM(Tonni_estratte) AS FLOAT) AS ton_estratte,

 -- Oro_grammi (-> 2.5)
CAST(TRIM(Oro_grammi) AS FLOAT) AS oro_g,

 -- Argento_grammi (-> 15.0)
CAST(TRIM(Argento_grammi) AS FLOAT) AS argento_g,

 -- Rame_kg (-> 120.0)
CAST(TRIM(Rame_kg) AS FLOAT) AS rame_Kg,

 -- Ore_lavorate (-> 720.0)
CAST(TRIM(Ore_lavorate) AS FLOAT) AS ore_lavorative,

Note
FROM Produzione_Grezza;
"""

# Creiamo la tabella temporanea con i dati puliti (usa cursor dalla cella precedente)
cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_temp AS " + q)

# Verifichiamo il contenuto
print("\nProduzione_Grezza --> Puliti!")
df_verifica = pd.read_sql_query("SELECT * FROM Campioni_puliti_temp", conn)
print(df_verifica)


Produzione_Grezza --> Puliti!
   id_mese        data sito_estrazione  ton_estratte  oro_g  argento_g  \
0    GEN23  2023-01-01            NORD        1250.0    2.5       15.0   
1    FEB23  2023-02-01            NORD        1350.0    2.8       18.0   
2    MAR23  2023-03-01             SUD        1100.0    1.8        0.0   
3    APR23  2023-04-01             SUD         950.0    2.1       12.0   
4    MAG23  2023-05-01             EST         880.0    0.0        8.5   
5    GIU23  2023-06-01             EST        1020.0    2.2       20.0   
6    LUG23  2023-07-01           OVEST         980.0    0.0        0.0   
7    AGO23  2023-08-01           OVEST        1150.0    2.4        0.0   
8    SET23  2023-09-01            NORD        1250.0    0.0       25.0   
9    OTT23  2023-10-01             SUD        1080.0    2.0        0.0   
10   NOV23  2023-11-01             EST        1120.0    0.0       15.0   
11   DIC23  2023-12-31            NORD        1300.0    2.3       22.0   

    ra

## 📊 Calcolo parametri statistici fondamentali:

### Dopo la pulizia, calcola per Oro_grammi:

Misure di centro:
* Media
* Mediana
* Moda

Misure di spread:
* Range (max - min)
* Varianza
* Deviazione standard
* Coefficiente di variazione (CV = std/media)

Per ogni sito:
* Media Oro per sito
* Media Argento per sito
* Media Rame per sito
* Produttività media (Tonni_estratte / Ore_lavorate)

In [9]:
q_media_mediana = """
-- Misure di centro
SELECT 
    AVG(oro_g) AS media_oro,
    (SELECT AVG(oro_g) 
     FROM (
         SELECT oro_g
         FROM Campioni_puliti_temp
         WHERE oro_g IS NOT NULL
         ORDER BY oro_g
         LIMIT 2 - (SELECT COUNT(*) FROM Campioni_puliti_temp WHERE oro_g IS NOT NULL) % 2
         OFFSET (SELECT (COUNT(*) - 1) / 2 
                 FROM Campioni_puliti_temp WHERE oro_g IS NOT NULL)
     )) AS mediana_oro
FROM Campioni_puliti_temp;
"""

q_moda = """
-- Moda (valore più frequente)
SELECT 
    oro_g AS moda_oro,
    COUNT(*) AS frequenza
FROM Campioni_puliti_temp
WHERE oro_g IS NOT NULL
GROUP BY oro_g
ORDER BY frequenza DESC
LIMIT 1;
"""

q_spread = """
-- Misure di spread
SELECT 
    MIN(oro_g) AS minimo_oro,
    MAX(oro_g) AS massimo_oro,
    MAX(oro_g) - MIN(oro_g) AS range_oro,
    AVG((oro_g - media_oro) * (oro_g - media_oro)) AS varianza_oro,
    SQRT(AVG((oro_g - media_oro) * (oro_g - media_oro))) AS deviazione_std_oro
FROM Campioni_puliti_temp, 
     (SELECT AVG(oro_g) AS media_oro FROM Campioni_puliti_temp);
"""

df1 = pd.read_sql_query(q_media_mediana, conn)
print(df1)
print("\n")

df2 = pd.read_sql_query(q_moda, conn)
print(df2)
print("\n")

df3 = pd.read_sql_query(q_spread, conn)
print(df3)

   media_oro  mediana_oro
0   1.508333         2.05


   moda_oro  frequenza
0       0.0          4


   minimo_oro  massimo_oro  range_oro  varianza_oro  deviazione_std_oro
0         0.0          2.8        2.8      1.194097            1.092748


### 🎯 "Produzione per sito" significa:
### Analizzare gli indicatori di produzione (quantità di oro, argento, rame, tonnellate, ore lavorate) suddivisi per ciascun sito di estrazione (NORD, SUD, EST, OVEST), per capire:
* Quale sito è più ricco di un determinato minerale
* Quale è più efficiente (tonnellate/ora)
* Quale ha produzione più regolare o più variabile

### Prova a scrivere le query per rispondere a queste domande:
* Quale sito produce più oro in media al mese?
* Quale sito ha la produttività più alta (tonni_estratte / ore_lavorate)?
* Quale sito estrae più minerali in totale (somma di oro+argento+rame)?

In [10]:
q_produzione = """
SELECT
    sito_estrazione,
    ROUND(AVG(oro_g), 2) AS media_oro,
    ROUND(SUM(ton_estratte) / SUM(ore_lavorative), 3) AS produttivita,
    ROUND(SUM(oro_g + argento_g + (rame_kg * 1000)), 2) AS tot_minerali_grammi
FROM Campioni_puliti_temp
GROUP BY sito_estrazione
ORDER BY media_oro DESC;
"""

df4 = pd.read_sql_query(q_produzione, conn)
print(df4)

  sito_estrazione  media_oro  produttivita  tot_minerali_grammi
0             SUD       1.97         1.573             310017.9
1            NORD       1.90         1.839             390087.6
2           OVEST       1.20         1.578             200002.4
3             EST       0.73         1.525             270045.7


### Prova a scrivere le query per rispondere a queste domande:¶
* Analisi della correlazione tra produttività e tenore di oro?
* Quale minerale (oro/argento/rame) è più correlato con l'efficienza?
* Analisi temporale (trend mensili per sito)?
* Analisi per minerale (oro vs argento vs rame)

In [11]:
q_pearson_oro = """
WITH 
-- STEP 1: Prepariamo i dati riga per riga (Mese per Mese)
dati_mensili AS (
    SELECT 
        oro_g,
        (ton_estratte / ore_lavorative) AS produttivita
    FROM Campioni_puliti_temp
),
-- STEP 2: Calcoliamo le medie globali (un'unica riga con i due valori medi)
medie_globali AS (
    SELECT 
        AVG(oro_g) AS m_oro,
        AVG(ton_estratte / ore_lavorative) AS m_prod
    FROM Campioni_puliti_temp
)
-- STEP 3: Applichiamo la formula di Pearson
SELECT 
    -- Numeratore: Somma dei prodotti degli scarti
    SUM((oro_g - m_oro) * (produttivita - m_prod)) / 
    -- Denominatore: Radice del prodotto delle somme degli scarti al quadrato
    SQRT(
        SUM((oro_g - m_oro) * (oro_g - m_oro)) * SUM((produttivita - m_prod) * (produttivita - m_prod))
    ) AS coefficiente_pearson_oro
FROM dati_mensili, medie_globali; -- Il CROSS JOIN avviene qui
"""

df5 = pd.read_sql_query(q_pearson_oro, conn)
print(df5)

q_pearson_argento = """
WITH 
-- STEP 1: Prepariamo i dati riga per riga (Mese per Mese)
dati_mensili AS (
    SELECT 
        argento_g,
        (ton_estratte / ore_lavorative) AS produttivita
    FROM Campioni_puliti_temp
),
-- STEP 2: Calcoliamo le medie globali (un'unica riga con i due valori medi)
medie_globali AS (
    SELECT 
        AVG(argento_g) AS m_argento,
        AVG(ton_estratte / ore_lavorative) AS m_prod
    FROM Campioni_puliti_temp
)
-- STEP 3: Applichiamo la formula di Pearson
SELECT 
    -- Numeratore: Somma dei prodotti degli scarti
    SUM((argento_g - m_argento) * (produttivita - m_prod)) / 
    -- Denominatore: Radice del prodotto delle somme degli scarti al quadrato
    SQRT(
        SUM((argento_g - m_argento) * (argento_g - m_argento)) * SUM((produttivita - m_prod) * (produttivita - m_prod))
    ) AS coefficiente_pearson_argento
FROM dati_mensili, medie_globali; -- Il CROSS JOIN avviene qui
"""

df6 = pd.read_sql_query(q_pearson_argento, conn)
print(df6)

q_pearson_rame = """
WITH 
-- STEP 1: Prepariamo i dati riga per riga (Mese per Mese)
dati_mensili AS (
    SELECT 
        rame_Kg,
        (ton_estratte / ore_lavorative) AS produttivita
    FROM Campioni_puliti_temp
),
-- STEP 2: Calcoliamo le medie globali (un'unica riga con i due valori medi)
medie_globali AS (
    SELECT 
        AVG(rame_Kg) AS m_rame,
        AVG(ton_estratte / ore_lavorative) AS m_prod
    FROM Campioni_puliti_temp
)
-- STEP 3: Applichiamo la formula di Pearson
SELECT 
    -- Numeratore: Somma dei prodotti degli scarti
    SUM((rame_Kg - m_rame) * (produttivita - m_prod)) / 
    -- Denominatore: Radice del prodotto delle somme degli scarti al quadrato
    SQRT(
        SUM((rame_Kg - m_rame) * (rame_Kg - m_rame)) * SUM((produttivita - m_prod) * (produttivita - m_prod))
    ) AS coefficiente_pearson_rame
FROM dati_mensili, medie_globali; -- Il CROSS JOIN avviene qui
"""

df7 = pd.read_sql_query(q_pearson_rame, conn)
print(df7)

   coefficiente_pearson_oro
0                  0.264199
   coefficiente_pearson_argento
0                      0.487194
   coefficiente_pearson_rame
0                  -0.046545


In [14]:
q_considerazioni = """
SELECT * 
FROM Campioni_puliti_temp
ORDER BY Sito_estrazione, data ASC;
"""

df8 = pd.read_sql_query(q_considerazioni, conn)
print(df8)

   id_mese        data sito_estrazione  ton_estratte  oro_g  argento_g  \
0    MAG23  2023-05-01             EST         880.0    0.0        8.5   
1    GIU23  2023-06-01             EST        1020.0    2.2       20.0   
2    NOV23  2023-11-01             EST        1120.0    0.0       15.0   
3    GEN23  2023-01-01            NORD        1250.0    2.5       15.0   
4    FEB23  2023-02-01            NORD        1350.0    2.8       18.0   
5    SET23  2023-09-01            NORD        1250.0    0.0       25.0   
6    DIC23  2023-12-31            NORD        1300.0    2.3       22.0   
7    LUG23  2023-07-01           OVEST         980.0    0.0        0.0   
8    AGO23  2023-08-01           OVEST        1150.0    2.4        0.0   
9    MAR23  2023-03-01             SUD        1100.0    1.8        0.0   
10   APR23  2023-04-01             SUD         950.0    2.1       12.0   
11   OTT23  2023-10-01             SUD        1080.0    2.0        0.0   

    rame_Kg  ore_lavorative          

### Analisi temporale (trend mensili per sito)
* il settore EST non mostra correlazioni significative tonnellate estratte / aumento di minerale per oro e rame, sembrano dipendere da depositi locali piuttosto che una continuita'. L'argento necessita' di ulteriori dati per la conferma.
* il settore NORD mostra correlazioni significative tonnellate estratte / aumento di minerale per tutti i minerali.
* il settore OVEST mostra totale assenza di argento.
* il settore SUD mostra continuita' nell'oro e depositi locali di rame e argento.

### 📈 Le tue considerazioni per sito:

Sito|Osservazione|Interpretazione geologica
|:---|:---|:---|
EST|Nessuna correlazione significativa|Mineralizzazione discontinua, depositi localizzati
NORD|Correlazione significativa per tutti|Zona più omogenea, produzione regolare
OVEST|Assenza totale di argento|Dominio mineralogico diverso (solo oro e rame)
SUD|Continuità nell'oro, depositi locali per Ag/Cu|Zona mista: oro diffuso, altri metalli in pockets

### 🎯 Riflessioni finali:
* Le tue analisi suggeriscono che:
* NORD è il sito più prevedibile e gestibile
* EST e SUD richiedono modelli geologici più dettagliati
* OVEST ha un potenziale specifico per oro/rame ma non argento